# Scraping de emergencias

Import packages

In [2]:
import sys, selenium
import pandas as pd
import requests, re, time
print("Python :", sys.executable)
print("Selenium :", selenium.__version__)

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

Python : c:\Users\User\anaconda3\python.exe
Selenium : 4.49.0


1. Descarguen e impriman el archivo https://www.gob.pe/robots.txt. En una celda de texto, expliquen qué prohíbe, si la página de búsquedas (/busquedas) está permitida y por qué igual conviene hacer pausas entre página y página (fíjense en el Crawl-delay que les piden a otros bots).

In [3]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3"
}

url_robots = "https://www.gob.pe/robots.txt"

robots = requests.get(url_robots, headers= headers, timeout=30)

print(robots.text)

# See http://www.robotstxt.org/robotstxt.html for documentation on how to use the robots.txt file
#
# To ban all spiders from the entire site uncomment the next two lines:
User-agent: *
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: AhrefsBot
Disallow: /

User-agent: GPTBot
Crawl-delay: 5
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: OAI-SearchBot
Crawl-delay: 2
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

Sitemap: https://www.gob.pe/sitemaps/sitemap.xml.gz



/busquedas no aparece entre las rutas prohibidas para User-agent: *, por lo que la página de búsquedas está permitida según este archivo. Para el bloque general User-agent: * no se establece un Crawl-delay. Los valores de 5 segundos para GPTBot y 2 segundos para OAI-SearchBot corresponden específicamente a esos bots y no son obligatorios para nuestro script. Aun así, se mantuvieron pausas entre solicitudes como una práctica responsable para evitar sobrecargar el servidor.

2. Con Selenium, abran la búsqueda de un solo mes de su temporada. Usen WebDriverWait, no solo time.sleep().

De la temporada asignada enero 1 de 2024 a mayo 31 de 2024, se toma el mes de enero. Por lo tanto, la busqueda debe retornar 1 de enero al 31 de enero de 2024.

In [4]:
navegador = webdriver.Chrome() # Sets Chrome as the browser

navegador.get("https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde=01-01-2024&hasta=31-01-2024")
# Opens browser to the website, searching for entries between Jan 1, 2024 and Jan 31, 2024

WebDriverWait(navegador, 30).until(EC.presence_of_all_elements_located((By.TAG_NAME, "article"))) # Waits until all articles are loaded

time.sleep(5) # Waits 5 seconds 

print(navegador.title)

n_resultados = navegador.find_elements(By.TAG_NAME, "article")

print("Número de resultados:", len(n_resultados))

Plataforma del Estado Peruano
Número de resultados: 11


3. Lean el número total de resultados que muestra la página ("N Resultados").

In [5]:
subt_n_resultados = navegador.find_element(By.TAG_NAME, "h2")
texto = subt_n_resultados.text
print(texto)

11 Resultados


In [6]:
palabras = texto.split()
print(palabras)

['11', 'Resultados']


In [7]:
resultados_totales_ene = int(palabras[0])
print(type(resultados_totales_ene))

<class 'int'>


Son 11 los resultados encontrados.

4. De cada resultado, extraigan:

número de la norma: es el texto del enlace principal (por ejemplo: Decreto Supremo N.° 044-2023-PCM);
fecha de publicación: el texto que empieza con Publicado:;
título: el párrafo debajo de la fecha;
enlace a la norma.

Número de la norma


In [8]:
print(n_resultados[0].text) 

PCM - Presidencia del Consejo de Ministros
Normas y documentos legales
›
Decretos
Decreto Supremo N.° 010-2024-PCM
Publicado: 30 de enero de 2024
Prorrogar por el termino de sesenta (60) días calendario a partir del 6 de febrero al 5 de abril de 2024, el Estado de Emergencia en los distritos que se indican en el Anexo que forma parte integrant ...
Descargar archivo


In [9]:
for etiqueta in ["a", "h4", "p", "span"]:
    encontrados = n_resultados[0].find_elements(By.TAG_NAME, etiqueta) 
    # Encuentra el # de resultados en el 1° elemento de n_resultados por cada etiqueta
    print(etiqueta, ":", len(encontrados))
    for elemento in encontrados:
        print(elemento.text)

a : 2
Decreto Supremo N.° 010-2024-PCM
Descargar archivo
h4 : 1
Decreto Supremo N.° 010-2024-PCM
p : 1
Prorrogar por el termino de sesenta (60) días calendario a partir del 6 de febrero al 5 de abril de 2024, el Estado de Emergencia en los distritos que se indican en el Anexo que forma parte integrant ...
span : 3
Normas y documentos legales
Decretos
Publicado: 30 de enero de 2024


Número de la norma es un <h4>

Fecha de publicación es un <span[2]>

Título es un <p> 

Enlace de la norma es <a[0]>

In [10]:
numero = n_resultados[0].find_element(By.TAG_NAME, "h4").text

fecha = n_resultados[0].find_elements(By.TAG_NAME, "span")[2].text

titulo = n_resultados[0].find_element(By.TAG_NAME, "p").text

enlace = n_resultados[0].find_element(By.TAG_NAME, "a").get_attribute("href")

print("Número: ", numero)
print("fecha: ", fecha)
print("titulo: ", titulo)
print("enlace: ", enlace)

Número:  Decreto Supremo N.° 010-2024-PCM
fecha:  Publicado: 30 de enero de 2024
titulo:  Prorrogar por el termino de sesenta (60) días calendario a partir del 6 de febrero al 5 de abril de 2024, el Estado de Emergencia en los distritos que se indican en el Anexo que forma parte integrant ...
enlace:  https://www.gob.pe/institucion/pcm/normas-legales/5398152-010-2024-pcm


Ahora se repite para todo enero

In [11]:
filas = [] # conjunto vacío para almacenar datos del artículo

for articulo in n_resultados: # por cada artículo en n_resultados
    numero = articulo.find_element(By.TAG_NAME, "h4").text
    fecha = articulo.find_elements(By.TAG_NAME, "span")[2].text
    titulo = articulo.find_element(By.TAG_NAME, "p").text
    enlace = articulo.find_element(By.TAG_NAME, "a").get_attribute("href")
    
    filas.append([numero, fecha, titulo, enlace]) # Se añade una fila con los datos del artículo abajo

df_enero = pd.DataFrame(filas, columns=["Número", "Fecha", "Título", "Enlace"]) 
# Se guarda en el data frame de enero 

df_enero

,Número,Fecha,Título,Enlace
0,Decreto Supremo N.° 010-2024-PCM,Publicado: 30 de enero de 2024,Prorrogar por el termino de sesenta (60) días ...,https://www.gob.pe/institucion/pcm/normas-lega...
1,Decreto Supremo N.° 009-2024-PCM,Publicado: 29 de enero de 2024,Prorrogar por el termino de sesenta (60) días ...,https://www.gob.pe/institucion/pcm/normas-lega...
2,Decreto Supremo N.° 008-2024-PCM,Publicado: 29 de enero de 2024,Declarar por el termino de treinta (30) días c...,https://www.gob.pe/institucion/pcm/normas-lega...
3,Decreto Supremo N.° 007-2024-PCM,Publicado: 29 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
4,Decreto Supremo N.° 006-2024-PCM,Publicado: 29 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
5,Decreto Supremo N.° 005-2024-PCM,Publicado: 29 de enero de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
6,Decreto Supremo N.° 004-2024-PCM,Publicado: 26 de enero de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
7,Resolución N.° 0214-2024-TCE-S6,Publicado: 18 de enero de 2024,N° 10304103618); por su responsabilidad al hab...,https://www.gob.pe/institucion/oece/normas-leg...
8,Decreto Supremo N.° 003-2024-PCM,Publicado: 10 de enero de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
9,Decreto Supremo N.° 002-2024-PCM,Publicado: 9 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...


5. Repitan los pasos 2 a 4 para todos los meses de su temporada. Hagan una pausa de al menos 2 segundos entre cada página.

Se definen los meses

In [12]:
meses = [("enero", "01-01-2024", "31-01-2024"),
         ("febrero", "01-02-2024", "29-02-2024"),
         ("marzo", "01-03-2024", "31-03-2024"),
         ("abril", "01-04-2024", "30-04-2024"),
         ("mayo", "01-05-2024", "31-05-2024")]

print(meses)

[('enero', '01-01-2024', '31-01-2024'), ('febrero', '01-02-2024', '29-02-2024'), ('marzo', '01-03-2024', '31-03-2024'), ('abril', '01-04-2024', '30-04-2024'), ('mayo', '01-05-2024', '31-05-2024')]


Se crean los urls:

In [13]:
for mes, desde, hasta in meses:
    print(f"https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde={desde}&hasta={hasta}")

https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde=01-01-2024&hasta=31-01-2024
https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde=01-02-2024&hasta=29-02-2024
https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde=01-03-2024&hasta=31-03-2024
https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde=01-04-2024&hasta=30-04-2024
https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde=01-05-2024&hasta=31-05-2024


In [14]:
filas = []
verificacion =[]

for mes, desde, hasta in meses:
    # Para cada mes, se arma el url
    url = f"https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde={desde}&hasta={hasta}"

    # Se usa Chrome como navegador
    navegador = webdriver.Chrome() 

    # Se abre Chrome con el url del mes asignado
    navegador.get(url)

    # Se espera a que los artículos carguen
    WebDriverWait(navegador, 30).until(EC.presence_of_all_elements_located((By.TAG_NAME, "article")))

    # Se guardan los resultados 
    n_resultados = navegador.find_elements(By.TAG_NAME, "article")

    # Para comparar el número de resultados obtenidos vs los declarados en la página, se guarda el # de resultados declarados por la página
    subt_n_resultados = navegador.find_element(By.TAG_NAME, "h2")
    texto = subt_n_resultados.text
    palabras = texto.split()
    resultados_totales = int(palabras[0])

    # Se guardan los datos del mes en el data frame
    for articulo in n_resultados: # por cada artículo en n_resultados
        numero = articulo.find_element(By.TAG_NAME, "h4").text
        fecha = articulo.find_elements(By.TAG_NAME, "span")[2].text
        titulo = articulo.find_element(By.TAG_NAME, "p").text
        enlace = articulo.find_element(By.TAG_NAME, "a").get_attribute("href")
    
        filas.append([mes, numero, fecha, titulo, enlace])

    # Se guarda el mes, resultados declarados y resultados obtenidos 
    verificacion.append([mes, resultados_totales, len(n_resultados)]) # Se guarda el mes, el número de resultados declarados y el número de resultados obtenidos

    time.sleep(2) # Se espera 2 segundos antes de repetirse el proceso

    navegador.quit() # Se cierra Chrome

df = pd.DataFrame(filas, columns=["mes", "numero", "fecha", "titulo", "enlace"]) 

df_verificacion = pd.DataFrame(verificacion, columns=["mes", "resultados_totales", "resultados_extraidos"])

df_verificacion

,mes,resultados_totales,resultados_extraidos
0,enero,11,11
1,febrero,11,11
2,marzo,11,11
3,abril,15,15
4,mayo,3,3


Los números coinciden.

7. Eliminen las normas repetidas.

In [15]:
before = len(df)
print(before)

df.head(10)

51


,mes,numero,fecha,titulo,enlace
0,enero,Decreto Supremo N.° 010-2024-PCM,Publicado: 30 de enero de 2024,Prorrogar por el termino de sesenta (60) días ...,https://www.gob.pe/institucion/pcm/normas-lega...
1,enero,Decreto Supremo N.° 009-2024-PCM,Publicado: 29 de enero de 2024,Prorrogar por el termino de sesenta (60) días ...,https://www.gob.pe/institucion/pcm/normas-lega...
2,enero,Decreto Supremo N.° 008-2024-PCM,Publicado: 29 de enero de 2024,Declarar por el termino de treinta (30) días c...,https://www.gob.pe/institucion/pcm/normas-lega...
3,enero,Decreto Supremo N.° 007-2024-PCM,Publicado: 29 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
4,enero,Decreto Supremo N.° 006-2024-PCM,Publicado: 29 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
5,enero,Decreto Supremo N.° 005-2024-PCM,Publicado: 29 de enero de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
6,enero,Decreto Supremo N.° 004-2024-PCM,Publicado: 26 de enero de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
7,enero,Resolución N.° 0214-2024-TCE-S6,Publicado: 18 de enero de 2024,N° 10304103618); por su responsabilidad al hab...,https://www.gob.pe/institucion/oece/normas-leg...
8,enero,Decreto Supremo N.° 003-2024-PCM,Publicado: 10 de enero de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
9,enero,Decreto Supremo N.° 002-2024-PCM,Publicado: 9 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...


In [16]:
df = df.drop_duplicates(subset = ["numero"]) 

after = len(df)
print(after)

df.head(10)

51


,mes,numero,fecha,titulo,enlace
0,enero,Decreto Supremo N.° 010-2024-PCM,Publicado: 30 de enero de 2024,Prorrogar por el termino de sesenta (60) días ...,https://www.gob.pe/institucion/pcm/normas-lega...
1,enero,Decreto Supremo N.° 009-2024-PCM,Publicado: 29 de enero de 2024,Prorrogar por el termino de sesenta (60) días ...,https://www.gob.pe/institucion/pcm/normas-lega...
2,enero,Decreto Supremo N.° 008-2024-PCM,Publicado: 29 de enero de 2024,Declarar por el termino de treinta (30) días c...,https://www.gob.pe/institucion/pcm/normas-lega...
3,enero,Decreto Supremo N.° 007-2024-PCM,Publicado: 29 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
4,enero,Decreto Supremo N.° 006-2024-PCM,Publicado: 29 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
5,enero,Decreto Supremo N.° 005-2024-PCM,Publicado: 29 de enero de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
6,enero,Decreto Supremo N.° 004-2024-PCM,Publicado: 26 de enero de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
7,enero,Resolución N.° 0214-2024-TCE-S6,Publicado: 18 de enero de 2024,N° 10304103618); por su responsabilidad al hab...,https://www.gob.pe/institucion/oece/normas-leg...
8,enero,Decreto Supremo N.° 003-2024-PCM,Publicado: 10 de enero de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
9,enero,Decreto Supremo N.° 002-2024-PCM,Publicado: 9 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...


In [17]:
print("Diferencia antes y después: ", before - after)

Diferencia antes y después:  0


No hubo diferencia antes y depués de la eliminación de duplicados. 

Ojo: aunque la búsqueda filtra por PCM, se cuelan normas de otras instituciones (por ejemplo, resoluciones del OECE). Quédense solo con las que tienen /institucion/pcm/ en el enlace e impriman cuántas quitaron.

In [18]:
df = df[df["enlace"].str.contains("institucion/pcm/")]

after = len(df)

print("Eliminadas: ", before - after)

Eliminadas:  8


9. Los títulos de la lista aparecen cortados ("..."), y justo lo que se corta suele ser el motivo ("por peligro inminente..."). La página de cada norma no necesita Selenium: léanla con requests y BeautifulSoup (Revisar Sesión 4) y saquen el título completo. Hagan una pausa de 1 segundo entre páginas.

Se toma la primera observación de ejemplo

In [19]:
url_test = df["enlace"].iloc[0] # enlace da la columna, iloc da la fila 

print(url_test)

https://www.gob.pe/institucion/pcm/normas-legales/5398152-010-2024-pcm


In [20]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3"
}

respuesta = requests.get(url_test, headers= headers, timeout=30)

print(respuesta.status_code)

200


In [21]:
from bs4 import BeautifulSoup # importa la libreria 

sopa = BeautifulSoup(respuesta.text, "html.parser") 

titulo_completo = sopa.find("div", "description").get_text(strip=True)

titulo_cortado = df["titulo"].iloc[0]

print("Título completo: ", titulo_completo)
print("Título cortado: ", titulo_cortado)

Título completo:  Prorrogar por el termino de sesenta (60) días calendario a partir del 6 de febrero al 5 de abril de 2024, el Estado de Emergencia en los distritos que se indican en el Anexo que forma parte integrante del presente Decreto Supremo.DS N° 0010-2024-PCMPDF1.9 MBDescargar
Título cortado:  Prorrogar por el termino de sesenta (60) días calendario a partir del 6 de febrero al 5 de abril de 2024, el Estado de Emergencia en los distritos que se indican en el Anexo que forma parte integrant ...


Se repite para todas las observaciones

In [22]:
titulos_completos = []

for enlace in df["enlace"]:
    respuesta = requests.get(enlace, headers= headers, timeout=30)
    sopa = BeautifulSoup(respuesta.text, "html.parser")

    titulo_completo = sopa.find("div", "description").get_text(strip=True)
    titulos_completos.append(titulo_completo)

    time.sleep(1) # Se espera 1 segundo antes de repetirse el proceso

df["titulo_completo"] = titulos_completos #Se añade columna al df

df[["titulo", "titulo_completo"]].head(10)


,titulo,titulo_completo
0,Prorrogar por el termino de sesenta (60) días ...,Prorrogar por el termino de sesenta (60) días ...
1,Prorrogar por el termino de sesenta (60) días ...,Prorrogar por el termino de sesenta (60) días ...
2,Declarar por el termino de treinta (30) días c...,Declarar por el termino de treinta (30) días c...
3,Decreto Supremo que prorroga el Estado de Emer...,Decreto Supremo que prorroga el Estado de Emer...
4,Decreto Supremo que prorroga el Estado de Emer...,Decreto Supremo que prorroga el Estado de Emer...
5,Decreto Supremo que declara el Estado de Emerg...,Decreto Supremo que declara el Estado de Emerg...
6,Decreto Supremo que declara el Estado de Emerg...,Decreto Supremo que declara el Estado de Emerg...
8,Decreto Supremo que declara el Estado de Emerg...,Decreto Supremo que declara el Estado de Emerg...
9,Decreto Supremo que prorroga el Estado de Emer...,Decreto Supremo que prorroga el Estado de Emer...
10,Decreto Supremo que prorroga el Estado de Emer...,Decreto Supremo que prorroga el Estado de Emer...


10. Creen estas columnas usando el título completo en minúsculas (.lower()) y el operador in:
    es_lluvia: True si el título contiene "lluvia" o "precipitaciones".
    
    tipo: "prorroga" si contiene "prórroga" o "prorroga"; si no, "declara" si contiene "declara"; si no, "otro".

    motivo: "peligro inminente" o "impacto de daños" si el título contiene esa frase; si no, "otro".

Revisen las filas que quedaron como "otro" y, en una celda de texto, expliquen si están bien clasificadas.

In [23]:
# Convertir a minúsculas
df["t_minus"] = df["titulo_completo"].str.lower()

# Definir función para clasificar
def clasifica_lluvia(texto):
    return "lluvia" in texto or "precipitaciones" in texto

# Aplicar la nueva función
df["es_lluvia"] = df["t_minus"].apply(clasifica_lluvia)

df["es_lluvia"].value_counts()

es_lluvia
False    23
True     20
Name: count, dtype: int64

In [24]:
# Definir función
def clasifica_tipo(texto):
    if "prórroga" in texto or "prorroga" in texto:
        return "prorroga"
    elif "declara" in texto:
        return "declara"
    else:
        return "otro"

df["tipo"] = df["t_minus"].apply(clasifica_tipo)

df["tipo"].value_counts()

tipo
prorroga    26
declara     17
Name: count, dtype: int64

In [25]:
# Definir función
def clasifica_motivo(texto):
    if "peligro inminente" in texto:
        return "peligro inminente"
    elif "impacto de daños" in texto:
        return "impacto de daños"
    else:
        return "otro"

df["motivo"] = df["t_minus"].apply(clasifica_motivo)

df["motivo"].value_counts()

motivo
otro                 21
impacto de daños     20
peligro inminente     2
Name: count, dtype: int64

In [26]:
otros = df[df["motivo"] == "otro"]

for t in otros["titulo_completo"]:
    print(t)
    print()


Prorrogar por el termino de sesenta (60) días calendario a partir del 6 de febrero al 5 de abril de 2024, el Estado de Emergencia en los distritos que se indican en el Anexo que forma parte integrante del presente Decreto Supremo.DS N° 0010-2024-PCMPDF1.9 MBDescargar

Prorrogar por el termino de sesenta (60) días calendario, a partir del 1 de febrero de 2024, el Estado de Emergencia declarado en los distritos de Tambopata, Inambari, Las Piedras y Laberinto de la provincia de Tambopata y en los distritos de Madre de Dios y Huepetuhe de la provincia de Manu del departamento de Madre de Dios. La Policía Nacional del Perú mantiene el control del orden interno, con el apoyo de las Fuerzas Armadas.DS N° 009-2024-PCMPDF1.2 MBDescargar

Declarar por el termino de treinta (30) días calendario, a partir del 5 de febrero de 2024 el Estado de Emergencia en el Corredor Vial Sur Apurímac-Cusco-Arequipa, incluyendo los quinientos (500) metros adyacentes a cada lado de la via. La Policía Nacional del 

In [27]:
otros["es_lluvia"].value_counts()

es_lluvia
False    21
Name: count, dtype: int64

Estas 21 observaciones clasificadas como otro suelen estar relacionadas al orden interno. Una limitación de usar el método in es que, para las observaciones en que no se explicita las motivaciones, es difícil asegurar si tienen o no relación con las lluvias

11. Quédense solo con las normas de lluvias. Impriman cuántas son y muestren 5 filas.

In [28]:
df_lluvias = df[df["es_lluvia"] == True]

print(len(df_lluvias))

df_lluvias.head(5)

20


,mes,numero,fecha,titulo,enlace,titulo_completo,t_minus,es_lluvia,tipo,motivo
3,enero,Decreto Supremo N.° 007-2024-PCM,Publicado: 29 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...,Decreto Supremo que prorroga el Estado de Emer...,decreto supremo que prorroga el estado de emer...,True,prorroga,impacto de daños
4,enero,Decreto Supremo N.° 006-2024-PCM,Publicado: 29 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...,Decreto Supremo que prorroga el Estado de Emer...,decreto supremo que prorroga el estado de emer...,True,prorroga,peligro inminente
5,enero,Decreto Supremo N.° 005-2024-PCM,Publicado: 29 de enero de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...,Decreto Supremo que declara el Estado de Emerg...,decreto supremo que declara el estado de emerg...,True,declara,impacto de daños
6,enero,Decreto Supremo N.° 004-2024-PCM,Publicado: 26 de enero de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...,Decreto Supremo que declara el Estado de Emerg...,decreto supremo que declara el estado de emerg...,True,declara,impacto de daños
11,febrero,Decreto Supremo N.° 021-2024-PCM,Publicado: 28 de febrero de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...,Decreto Supremo que declara el Estado de Emerg...,decreto supremo que declara el estado de emerg...,True,declara,impacto de daños


12. Identifiquen qué departamentos menciona cada decreto. Un decreto puede mencionar varios. 

In [29]:
departamentos = ["Amazonas", "Áncash", "Apurímac", "Arequipa", "Ayacucho",
"Cajamarca", "Callao", "Cusco", "Huancavelica", "Huánuco",
"Ica", "Junín", "La Libertad", "Lambayeque", "Lima", "Loreto",
"Madre de Dios", "Moquegua", "Pasco", "Piura", "Puno",
"San Martín", "Tacna", "Tumbes", "Ucayali"]

In [30]:
titulo = df_lluvias["titulo_completo"].iloc[0]

encontrados = []

for departamento in departamentos:
    if re.search(r"\b" + departamento + r"\b", titulo):
        encontrados.append(departamento)

print(titulo)
print(encontrados)

Decreto Supremo que prorroga el Estado de Emergencia en el distrito de Echarate de la provincia de La Convención del departamento de Cusco, por impacto de daños a consecuencia de intensas precipitaciones pluvialesDS N° 007-2024-PCMPDF1.5 MBDescargar
['Cusco']


In [31]:
def buscar_departamentos(titulo):
    encontrados = []

    aliases = {
        "Áncash": ["Áncash", "Ancash"],
        "San Martín": ["San Martín", "San Martin"]
    }

    for departamento in departamentos:
        variantes = aliases.get(departamento, [departamento])

        if any(
            re.search(r"\b" + re.escape(variante) + r"\b", titulo)
            for variante in variantes
        ):
            encontrados.append(departamento)

    return encontrados

df_lluvias["departamentos"] = df_lluvias["titulo_completo"].apply(buscar_departamentos)

df_lluvias[["numero", "departamentos"]]

,numero,departamentos
3,Decreto Supremo N.° 007-2024-PCM,[Cusco]
4,Decreto Supremo N.° 006-2024-PCM,"[Amazonas, Áncash, Arequipa, Ayacucho, Cajamar..."
5,Decreto Supremo N.° 005-2024-PCM,"[Arequipa, Ayacucho, Cusco, Huancavelica, Huán..."
6,Decreto Supremo N.° 004-2024-PCM,[Apurímac]
11,Decreto Supremo N.° 021-2024-PCM,"[Arequipa, Ica, Madre de Dios, Moquegua, Ucayali]"
12,Decreto Supremo N.° 020-2024-PCM,"[Amazonas, Áncash, Apurímac, Arequipa, Ayacuch..."
18,Decreto Supremo N.° 016-2024-PCM,"[Amazonas, Áncash, Apurímac, Arequipa, Ayacuch..."
21,Decreto Supremo N.° 0012-2024-PCM,"[Áncash, Apurímac, Cusco, Huancavelica, Lima, ..."
23,Decreto Supremo N.° 033-2024-PCM,"[Arequipa, Ayacucho, Cusco, Huancavelica, Huán..."
24,Decreto Supremo N.° 032-2024-PCM,"[Áncash, Apurímac, Cusco, Huancavelica, Lima, ..."


Verificar Ica

In [32]:
verificar_ica = df_lluvias["titulo_completo"].str.contains("Ica")

verificar_ica.value_counts()

titulo_completo
False    13
True      7
Name: count, dtype: int64

In [33]:
con_ica = df_lluvias[verificar_ica]

for num, deps, t in zip(con_ica["numero"], con_ica["departamentos"], con_ica["titulo_completo"]):
    print(num)
    print("Detectados:", deps)
    print(t)
    print()

Decreto Supremo N.° 006-2024-PCM
Detectados: ['Amazonas', 'Áncash', 'Arequipa', 'Ayacucho', 'Cajamarca', 'Ica', 'La Libertad', 'Lambayeque', 'Lima', 'Pasco', 'Piura', 'San Martín', 'Tumbes']
Decreto Supremo que prorroga el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Arequipa, Ayacucho, Cajamarca, Ica, La Libertad, Lambayeque, Lima, Pasco, Piura, San Martín y Tumbes, por peligro inminente ante intensas precipitaciones pluviales (período 2023- 2024) y posible Fenómeno El NiñoDS N° 006-2024-PCMPDF3.6 MBDescargar

Decreto Supremo N.° 005-2024-PCM
Detectados: ['Arequipa', 'Ayacucho', 'Cusco', 'Huancavelica', 'Huánuco', 'Ica', 'Moquegua', 'Pasco']
Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Arequipa, Ayacucho, Cusco, Huánuco, Huancavelica, Ica, Moquegua y Pasco, por impacto de daños a consecuencia de intensas precipitaciones pluvialesDS N° 005-2024-PCMPDF2 MB

La función utilizada sí logra detectar cuando las menciones de Ica corresponden al departamento de Ica y no a errores como considerar Huancavelica. Esto se debe a la capitaclización de la I, que no se encuntra en Huancavelica. Por ello, se busca en los títulos originales, y no en la columna de títulos en que todo se pasó a minúsculas. 

13. Guarden dos archivos:
    datos/decretos_lluvias.csv con las columnas numero, fecha, titulo, tipo, motivo y enlace.
    datos/decretos_por_departamento.csv con las columnas departamento, declaratorias y prorrogas.

In [34]:
decretos_lluvias = df_lluvias[["numero", "fecha", "titulo_completo", "tipo", "motivo", "enlace"]]
decretos_lluvias = decretos_lluvias.rename(columns = {"titulo_completo" : "titulo"})

decretos_lluvias.to_csv("datos/decretos_lluvias.csv", index=False, encoding="utf-8-sig")



In [35]:
df_largo = (
    df_lluvias
    .explode("departamentos")
    .reset_index(drop=True)
)

df_largo.head(5)


,mes,numero,fecha,titulo,enlace,titulo_completo,t_minus,es_lluvia,tipo,motivo,departamentos
0,enero,Decreto Supremo N.° 007-2024-PCM,Publicado: 29 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...,Decreto Supremo que prorroga el Estado de Emer...,decreto supremo que prorroga el estado de emer...,True,prorroga,impacto de daños,Cusco
1,enero,Decreto Supremo N.° 006-2024-PCM,Publicado: 29 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...,Decreto Supremo que prorroga el Estado de Emer...,decreto supremo que prorroga el estado de emer...,True,prorroga,peligro inminente,Amazonas
2,enero,Decreto Supremo N.° 006-2024-PCM,Publicado: 29 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...,Decreto Supremo que prorroga el Estado de Emer...,decreto supremo que prorroga el estado de emer...,True,prorroga,peligro inminente,Áncash
3,enero,Decreto Supremo N.° 006-2024-PCM,Publicado: 29 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...,Decreto Supremo que prorroga el Estado de Emer...,decreto supremo que prorroga el estado de emer...,True,prorroga,peligro inminente,Arequipa
4,enero,Decreto Supremo N.° 006-2024-PCM,Publicado: 29 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...,Decreto Supremo que prorroga el Estado de Emer...,decreto supremo que prorroga el estado de emer...,True,prorroga,peligro inminente,Ayacucho


In [36]:
decretos_por_departamento = pd.crosstab(df_largo["departamentos"], df_largo["tipo"])
decretos_por_departamento = decretos_por_departamento.reset_index()
decretos_por_departamento = decretos_por_departamento.rename(columns =  {"departamentos" : "departamento",
                                                                        "declara" : "declaratorias",
                                                                        "prorroga" : "prorrogas"})
decretos_por_departamento

tipo,departamento,declaratorias,prorrogas
0,Amazonas,4,2
1,Apurímac,7,4
2,Arequipa,7,6
3,Ayacucho,7,5
4,Cajamarca,3,3
5,Cusco,7,5
6,Huancavelica,7,4
7,Huánuco,6,3
8,Ica,4,3
9,Junín,5,1


In [37]:
decretos_por_departamento = decretos_por_departamento[["departamento", "declaratorias", "prorrogas"]]

decretos_por_departamento.to_csv("datos/decretos_por_departamento.csv", index=False, encoding="utf-8-sig")